# Benchmarking: Read directly from link

This notebooks reads EOPF Zarr and SAFE files directly to benchmark the performance when opening the data.
- EOPF Zarr from EODC via http
- SAFE from EODC via http
- SAFE from CDSE via S3

Potential file lists from this [notebook](https://github.com/xcube-dev/xcube-stac/blob/main/examples/notebooks/sentinel_2_cdse.ipynb) and this [notebook](https://eopf-sample-service.github.io/eopf-sample-notebooks/xcube-eopf-sen2/).

eopf:

browser:
- https://stac.browser.user.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316?.language=de
- https://stac.browser.user.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558?.language=de
- https://stac.browser.user.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207?.language=de

items:
- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316
- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558
- https://stac.core.eopf.eodc.eu/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207

cdse equivalents: 

browser:
- https://browser.stac.dataspace.copernicus.eu/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316?.language=de
- https://browser.stac.dataspace.copernicus.eu/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558?.language=de
- https://browser.stac.dataspace.copernicus.eu/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207?.language=de

items:
- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316
- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2C_MSIL2A_20250501T104041_N0511_R008_T32UNE_20250501T161558
- https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a/items/S2B_MSIL2A_20250506T103629_N0511_R008_T32UNE_20250506T115207

## Libraries

In [1]:
import xarray as xr
import dask
import time
import logging

#for direct loading
import rioxarray
import fsspec
import s3fs

## Define paths to files

In [2]:
#path_eopf_zarr = "https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:sample-data/tutorial_data/cpm_v253/S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458.zarr"
path_eodc_zarr = "https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/03/products/cpm_v256/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.zarr"
path_eodc_safe = "https://objects.eodc.eu/e05ab01a9d56408d82ac32d69a5aae2a:notebook-data/SAFE/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE"
path_cdse_safe = "s3://eodata/Sentinel-2/MSI/L2A/2025/05/03/S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE"

## Set up logger for counting http requests

In [3]:
# Silent in-memory log capture 
fsspec_logs = []

class ListHandler(logging.Handler):
    def __init__(self, storage):
        super().__init__()
        self.storage = storage
    def emit(self, record):
        self.storage.append(self.format(record))

# Disable root logger output to notebook
logging.getLogger().handlers.clear()

# Configure only fsspec.http logger
logger_fsspec = logging.getLogger("fsspec.http")
logger_fsspec.handlers.clear()
logger_fsspec.propagate = False  # <-- important! stops logs bubbling up
logger_fsspec.setLevel(logging.DEBUG)
logger_fsspec.addHandler(ListHandler(fsspec_logs))


## Benchmarking

### EOPF Zarr on EODC

In [4]:
%%time
fsspec_logs.clear()
dt = xr.open_datatree(path_eodc_zarr, engine="zarr", chunks={})
print("HTTP requests:", len(fsspec_logs))

HTTP requests: 59
CPU times: user 697 ms, sys: 55.9 ms, total: 752 ms
Wall time: 1.01 s


In [5]:
%%time
fsspec_logs.clear()
band_eodc_zarr = dt["measurements/reflectance/r10m"]["b04"].load()
print("HTTP requests:", len(fsspec_logs))

HTTP requests: 36
CPU times: user 1.19 s, sys: 295 ms, total: 1.48 s
Wall time: 862 ms


In [6]:
band_eodc_zarr

<xarray.DataArray 'b04' (y: 10980, x: 10980)> Size: 964MB
array([[0.0818, 0.0915, 0.0966, ...,    nan,    nan,    nan],
       [0.0739, 0.1026, 0.1208, ...,    nan,    nan,    nan],
       [0.069 , 0.1102, 0.1338, ...,    nan,    nan,    nan],
       ...,
       [0.2148, 0.2218, 0.2404, ...,    nan,    nan,    nan],
       [0.1872, 0.2032, 0.2386, ...,    nan,    nan,    nan],
       [0.1754, 0.1932, 0.2158, ...,    nan,    nan,    nan]])
Coordinates:
  * x        (x) int64 88kB 499985 499995 500005 500015 ... 609755 609765 609775
  * y        (y) int64 88kB 5999995 5999985 5999975 ... 5890225 5890215 5890205
Attributes:
    _eopf_attrs:     {'add_offset': -0.1, 'coordinates': ['x', 'y'], 'dimensi...
    dtype:           <u2
    fill_value:      0
    long_name:       BOA reflectance from MSI acquisition at spectral band b0...
    proj:bbox:       [499980.0, 5890200.0, 609780.0, 6000000.0]
    proj:epsg:       32632
    proj:shape:      [10980, 10980]
    proj:transform:  [10.0, 0.0, 499980.0, 0.0, -10.0, 6000000.0, 0.0, 0.0, 1.0]
    proj:wkt2:       PROJCS["WGS 84 / UTM zone 32N",GEOGCS["WGS 84",DATUM["WG...
    units:           digital_counts
    valid_max:       65535
    valid_min:       1

### EOPF SAFE on EODC

In [7]:
# Full URL to the B04 10m band file, from f"{path_eopf_zarr}/manifest.safe"
b04_url = (
    f"{path_eodc_safe}/GRANULE/L2A_T32UNE_A051514_20250503T103937/IMG_DATA/R10m/"
    "T32UNE_20250503T103701_B04_10m.jp2"
)

In [8]:
%%time
fsspec_logs.clear()

# Open and read the band
fs = fsspec.filesystem("http")
with fs.open(b04_url) as f:
    band_eodc_safe = rioxarray.open_rasterio(f, masked=True)

print("HTTP requests:", len(fsspec_logs))

HTTP requests: 2
CPU times: user 85.8 ms, sys: 43.4 ms, total: 129 ms
Wall time: 186 ms


In [9]:
band_eodc_safe

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 482MB
[120560400 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 6e+06 6e+06 6e+06 ... 5.89e+06 5.89e+06
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:  1.0
    add_offset:    0.0

### SAFE on CDSE S3

In [10]:
# cdse credentials
credentials = {
    "key": "FTE4ZT820RDZTHOU6I8C",
    "secret": "EdSaK2k1DjJm1rTlbucDaaSsmSSawWFz9da9Wemz",
}

In [11]:
# S3 
fs = s3fs.S3FileSystem(
    key=credentials["key"],
    secret=credentials["secret"],
    client_kwargs={
        "region_name": "eu-central-1",
        "endpoint_url": "https://s3.dataspace.copernicus.eu"
    }
)

In [12]:
# Correct path from manifest.safe
band_path = (
    "eodata/Sentinel-2/MSI/L2A/2025/05/03/"
    "S2A_MSIL2A_20250503T103701_N0511_R008_T32UNE_20250503T173316.SAFE/"
    "GRANULE/L2A_T32UNE_A051514_20250503T103937/IMG_DATA/R10m/"
    "T32UNE_20250503T103701_B04_10m.jp2"
)

In [13]:
%%time
# Open the file from S3
with fs.open(band_path, mode="rb") as f:
    band_cdse_safe = rioxarray.open_rasterio(f, masked=True)

CPU times: user 155 ms, sys: 78.4 ms, total: 234 ms
Wall time: 1.38 s


In [14]:
band_cdse_safe

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 482MB
[120560400 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 6e+06 6e+06 6e+06 ... 5.89e+06 5.89e+06
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:  1.0
    add_offset:    0.0